# Laboratório de Ciência de Dados
## Aula 5 — Avaliação prática dos modelos: métricas de desempenho, validação cruzada e ajuste de hiperparâmetros

### Situação

Nas aulas anteriores, a equipe da **VGAbank** construiu modelos para detectar fraudes e comparou uma Árvore de Decisão, uma Random Forest e um Gradient Boosting.

Na Aula 4 surgiu uma questão importante:

> **Se alteramos o modelo e seus hiperparâmetros olhando repetidamente para o conjunto de teste, esse conjunto continua sendo uma avaliação realmente independente?**

Nesta aula, vamos organizar a avaliação de forma mais rigorosa.

### O que você vai fazer

Você irá:

- revisar métricas de classificação no contexto de um problema desbalanceado;
- observar por que uma única divisão treino/teste pode produzir uma visão limitada;
- aplicar **validação cruzada estratificada**;
- comparar modelos usando média e variabilidade das métricas;
- ajustar hiperparâmetros com **GridSearchCV**;
- conhecer a lógica do **RandomizedSearchCV**;
- avaliar o modelo escolhido **uma única vez** no conjunto de teste reservado.

A ideia central da aula será:

> **treino/desenvolvimento → validação cruzada → ajuste → escolha do modelo → teste final**


# 1. Preparando o ambiente

Utilizaremos `pandas`, `numpy`, `matplotlib` e componentes do `scikit-learn`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay
)

RANDOM_STATE = 42


# 2. Reconstruindo o problema da VGAbank

Utilizaremos o mesmo tipo de conjunto sintético das Aulas 3 e 4:

- 20.000 transações;
- 12 atributos numéricos;
- classe `0`: transação legítima;
- classe `1`: fraude;
- aproximadamente 2% de fraudes.

Manter o mesmo problema permite concentrar a atenção na **metodologia de avaliação**, e não em uma nova base de dados.


In [ ]:
X_array, y_array = make_classification(
    n_samples=20000,
    n_features=12,
    n_informative=7,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    weights=[0.98, 0.02],
    class_sep=1.15,
    flip_y=0.004,
    random_state=RANDOM_STATE
)

nomes_atributos = [f"atributo_{i}" for i in range(1, 13)]

X = pd.DataFrame(X_array, columns=nomes_atributos)
y = pd.Series(y_array, name="fraude")

pd.DataFrame({
    "quantidade": y.value_counts().sort_index(),
    "percentual": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})


### Pergunta 1

Por que **PR-AUC, recall e F1** podem ser mais informativos do que apenas acurácia neste problema?

Não é necessário escrever uma definição formal. Relacione sua resposta ao fato de que fraudes são raras.


# 3. Um conjunto que não será usado para escolher o modelo

Agora adotaremos uma regra importante.

Separaremos os dados em:

- **80%: desenvolvimento** — utilizado para treinar, comparar modelos e ajustar hiperparâmetros;
- **20%: teste final** — utilizado somente depois que todas as decisões estiverem tomadas.

O conjunto de teste será nosso equivalente a um **envelope lacrado**.


In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Desenvolvimento:", X_dev.shape)
print("Teste final:", X_test.shape)

print("\nPercentual de fraudes:")
print("Desenvolvimento:", round(y_dev.mean() * 100, 2), "%")
print("Teste final:", round(y_test.mean() * 100, 2), "%")


> **Regra desta aula:** até a seção final, não utilizaremos `X_test` e `y_test` para escolher modelos ou hiperparâmetros.


# 4. Métricas: o que significa um modelo ser “bom”?

Na Aula 4 utilizamos várias métricas. Agora elas serão usadas como **critérios de avaliação sistemática**.

Vamos relembrar o significado prático:

| Métrica | Pergunta que responde |
|---|---|
| Acurácia | Qual proporção total foi classificada corretamente? |
| Precisão | Entre os alertas de fraude, quantos eram realmente fraude? |
| Recall | Entre todas as fraudes, quantas foram encontradas? |
| F1 | Como equilibrar precisão e recall em um único valor? |
| ROC-AUC | Quão bem o modelo ordena/separa as classes em diferentes thresholds? |
| PR-AUC | Como precisão e recall se comportam quando a classe positiva é rara? |

Neste problema, usaremos **PR-AUC como métrica principal de comparação**.

Isso não significa que as demais métricas deixem de importar.


# 5. O problema de uma única divisão dos dados

Uma prática comum é dividir os dados uma vez em treino e validação.

Mas existe uma pergunta:

> **O resultado mudaria se outras observações caíssem no conjunto de validação?**

Vamos experimentar.

Treinaremos o mesmo Random Forest várias vezes, alterando somente a divisão entre treino e validação dentro do conjunto de desenvolvimento.


In [ ]:
resultados_divisoes = []

for seed in [1, 7, 21, 42, 99, 123, 2026]:
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev,
        y_dev,
        test_size=0.25,
        stratify=y_dev,
        random_state=seed
    )

    modelo = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    modelo.fit(X_train, y_train)

    prob = modelo.predict_proba(X_val)[:, 1]
    pred = modelo.predict(X_val)

    resultados_divisoes.append({
        "random_state_divisao": seed,
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred),
        "pr_auc": average_precision_score(y_val, prob)
    })

df_divisoes = pd.DataFrame(resultados_divisoes)
df_divisoes.round(4)


In [ ]:
df_divisoes[["recall", "f1", "pr_auc"]].agg(
    ["mean", "std", "min", "max"]
).round(4)


### Pergunta 2

Os resultados foram exatamente iguais em todas as divisões?

Se não foram, qual conclusão podemos tirar sobre uma avaliação baseada em **uma única separação treino/validação**?


# 6. Validação cruzada

A **validação cruzada** reduz nossa dependência de uma única divisão.

Na validação cruzada com `k` partes, ou **k-fold**:

1. dividimos os dados em `k` partes;
2. treinamos em `k - 1` partes;
3. avaliamos na parte restante;
4. repetimos o processo até que todas as partes tenham sido utilizadas para validação.

Nesta aula utilizaremos **5 folds**.


## 6.1 Por que `StratifiedKFold`?

Nosso problema é fortemente desbalanceado.

Por isso utilizaremos:

```python
StratifiedKFold
```

A estratificação procura manter aproximadamente a mesma proporção entre classes em cada fold.

Isso é especialmente importante quando a classe positiva é rara.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv)


# 7. Avaliando um modelo com várias métricas

Vamos avaliar uma Random Forest em cinco folds.

O modelo será treinado cinco vezes, cada vez com uma parte diferente dos dados utilizada para validação.


In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

metricas = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

cv_rf = cross_validate(
    rf_base,
    X_dev,
    y_dev,
    cv=cv,
    scoring=metricas,
    n_jobs=-1,
    return_train_score=False
)

cv_rf


O objeto retornado contém uma medida para cada fold.

Vamos organizá-las em uma tabela.


In [ ]:
resultados_rf = pd.DataFrame({
    "fold": range(1, 6),
    "acuracia": cv_rf["test_accuracy"],
    "precisao": cv_rf["test_precision"],
    "recall": cv_rf["test_recall"],
    "f1": cv_rf["test_f1"],
    "roc_auc": cv_rf["test_roc_auc"],
    "pr_auc": cv_rf["test_pr_auc"]
})

resultados_rf.round(4)


In [ ]:
resumo_rf = resultados_rf.drop(columns="fold").agg(["mean", "std"])
resumo_rf.round(4)


## Média não é tudo

Observe que agora não temos apenas:

> PR-AUC = 0,XX

Temos algo mais informativo:

> **PR-AUC média + variabilidade entre folds**

O desvio padrão ajuda a perceber quanto o resultado varia entre diferentes subconjuntos dos dados.

Um modelo com média ligeiramente maior, mas enorme instabilidade, pode exigir uma análise mais cuidadosa.


### Pergunta 3

Observe a tabela dos cinco folds.

Qual métrica apresentou maior variação?

Por que apresentar **média e desvio padrão** é mais informativo do que apresentar somente o melhor fold?


# 8. Comparando modelos corretamente

Agora vamos repetir a validação cruzada para os três modelos estudados na Aula 4:

- Árvore de Decisão;
- Random Forest;
- Gradient Boosting.

Todos serão avaliados usando **os mesmos folds**.


In [ ]:
modelos = {
    "Árvore de Decisão": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

comparacao = []

for nome, modelo in modelos.items():
    resultado = cross_validate(
        modelo,
        X_dev,
        y_dev,
        cv=cv,
        scoring={
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc",
            "pr_auc": "average_precision"
        },
        n_jobs=-1
    )

    comparacao.append({
        "modelo": nome,
        "recall_medio": resultado["test_recall"].mean(),
        "recall_dp": resultado["test_recall"].std(),
        "f1_medio": resultado["test_f1"].mean(),
        "f1_dp": resultado["test_f1"].std(),
        "roc_auc_medio": resultado["test_roc_auc"].mean(),
        "pr_auc_media": resultado["test_pr_auc"].mean(),
        "pr_auc_dp": resultado["test_pr_auc"].std()
    })

df_comparacao = pd.DataFrame(comparacao)
df_comparacao.round(4)


### Pergunta 4

Considerando **PR-AUC média** como critério principal:

1. Qual modelo apresentou o melhor resultado?
2. A diferença para o segundo colocado parece grande ou pequena?
3. Os desvios padrão são semelhantes?
4. A conclusão é necessariamente a mesma se utilizarmos recall como prioridade?

Essa análise mostra por que “qual é o melhor modelo?” é uma pergunta incompleta sem especificar **qual objetivo e qual métrica**.


# 9. Hiperparâmetros

Até agora, fornecemos valores como:

```python
n_estimators=100
max_depth=3
learning_rate=0.05
```

Esses valores não são aprendidos diretamente durante o treinamento.

Eles são **hiperparâmetros**: configurações que controlam o comportamento do algoritmo.

Exemplos na Random Forest:

- `n_estimators`: número de árvores;
- `max_depth`: profundidade máxima das árvores;
- `min_samples_leaf`: quantidade mínima de amostras permitida em uma folha;
- `max_features`: quantidade de atributos considerados em cada divisão.

A pergunta agora é:

> **Como escolher essas configurações sem olhar para o conjunto de teste?**


# 10. Grid Search

O `GridSearchCV` testa sistematicamente as combinações fornecidas e avalia cada configuração utilizando validação cruzada.

Vamos ajustar uma Random Forest.


In [ ]:
grade_parametros = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5]
}

grade_parametros


### Antes de executar

Quantas combinações existem?

Temos:

- 2 valores de `n_estimators`;
- 3 valores de `max_depth`;
- 2 valores de `min_samples_leaf`.

Portanto:

**2 × 3 × 2 = 12 combinações**

Como utilizaremos 5 folds:

**12 × 5 = 60 treinamentos**

Isso ajuda a compreender por que buscas de hiperparâmetros podem ter custo computacional elevado.


In [ ]:
rf_grid_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

grid = GridSearchCV(
    estimator=rf_grid_base,
    param_grid=grade_parametros,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_dev, y_dev)

print("Melhores hiperparâmetros:")
print(grid.best_params_)

print("\nMelhor PR-AUC média na validação cruzada:")
print(round(grid.best_score_, 4))


## 10.1 Não observe apenas o vencedor

O Grid Search armazena os resultados de todas as combinações.

Vamos inspecioná-los.


In [ ]:
resultados_grid = pd.DataFrame(grid.cv_results_)

colunas = [
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "rank_test_score"
]

resultados_grid[colunas].sort_values("rank_test_score").head(12).round(4)


### Pergunta 5

Compare as primeiras posições.

O melhor conjunto de hiperparâmetros venceu por uma diferença expressiva?

Há configurações praticamente empatadas?

Por que isso pode ser importante antes de escolher um modelo muito mais complexo ou computacionalmente caro?


# 11. O ajuste realmente melhorou o modelo?

Vamos comparar:

- Random Forest padrão utilizada anteriormente;
- melhor Random Forest encontrada pelo Grid Search.

Importante: ainda **não usaremos o conjunto de teste**.

A comparação continua baseada na validação cruzada.


In [ ]:
pr_auc_base = df_comparacao.loc[
    df_comparacao["modelo"] == "Random Forest",
    "pr_auc_media"
].iloc[0]

print("Random Forest base - PR-AUC média:",
      round(pr_auc_base, 4))

print("Random Forest ajustada - PR-AUC média:",
      round(grid.best_score_, 4))

print("Diferença:",
      round(grid.best_score_ - pr_auc_base, 4))


### Pergunta 6

O ajuste de hiperparâmetros produziu melhoria?

Se sim, ela foi grande ou pequena?

> Ajustar hiperparâmetros não garante uma melhoria espetacular. O objetivo é encontrar uma configuração mais adequada de forma metodologicamente correta.


# 12. Randomized Search

O Grid Search testa **todas** as combinações especificadas.

Quando existem muitos hiperparâmetros e muitos valores possíveis, isso pode se tornar caro.

Uma alternativa é o `RandomizedSearchCV`.

Ele experimenta apenas uma quantidade definida de combinações sorteadas de um espaço de busca.


In [ ]:
distribuicoes_parametros = {
    "n_estimators": [50, 100, 150, 200, 300],
    "max_depth": [4, 6, 8, 10, 15, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=distribuicoes_parametros,
    n_iter=8,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_search.fit(X_dev, y_dev)

print("Melhores hiperparâmetros:")
print(random_search.best_params_)

print("\nMelhor PR-AUC média:")
print(round(random_search.best_score_, 4))


### Grid Search × Randomized Search

**Grid Search**

Testa todas as combinações explicitamente definidas.

**Randomized Search**

Testa apenas uma quantidade definida de combinações.

O Randomized Search não é automaticamente melhor. Ele é especialmente útil quando o espaço de busca é grande e testar todas as possibilidades seria caro.


### Pergunta 7

Neste exemplo, qual método encontrou a maior PR-AUC média?

Isso é suficiente para afirmar que um método de busca é sempre superior ao outro?

Explique brevemente.


# 13. Uma armadilha: ajustar ao conjunto de validação

Mesmo utilizando validação cruzada, estamos testando várias possibilidades e escolhendo aquela que teve melhor desempenho no conjunto de desenvolvimento.

Por isso ainda precisamos de uma avaliação final independente.

É aqui que entra o conjunto de teste que reservamos no início.

> O teste final deve responder: **como o modelo escolhido se comporta em dados que não participaram das nossas decisões?**


# 14. Escolhendo o modelo antes de abrir o teste

Para esta atividade, adotaremos a seguinte regra:

- entre Grid Search e Randomized Search, escolheremos o modelo com maior **PR-AUC média na validação cruzada**.

Somente depois dessa decisão utilizaremos o conjunto de teste.


In [ ]:
if grid.best_score_ >= random_search.best_score_:
    modelo_final = grid.best_estimator_
    origem_modelo = "GridSearchCV"
    score_cv_final = grid.best_score_
else:
    modelo_final = random_search.best_estimator_
    origem_modelo = "RandomizedSearchCV"
    score_cv_final = random_search.best_score_

print("Modelo escolhido por:", origem_modelo)
print("PR-AUC média na validação cruzada:", round(score_cv_final, 4))
print("Configuração:")
print(modelo_final)


# 15. Finalmente: avaliação no conjunto de teste

Agora, e somente agora, utilizaremos `X_test` e `y_test`.

O melhor estimador já foi reajustado pelo mecanismo de busca usando o conjunto de desenvolvimento completo.

Vamos medir seu desempenho final.


In [ ]:
y_pred_final = modelo_final.predict(X_test)
y_prob_final = modelo_final.predict_proba(X_test)[:, 1]

resultado_final = pd.DataFrame({
    "metrica": [
        "Acurácia",
        "Precisão",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ],
    "valor": [
        accuracy_score(y_test, y_pred_final),
        precision_score(y_test, y_pred_final, zero_division=0),
        recall_score(y_test, y_pred_final, zero_division=0),
        f1_score(y_test, y_pred_final, zero_division=0),
        roc_auc_score(y_test, y_prob_final),
        average_precision_score(y_test, y_prob_final)
    ]
})

resultado_final.round(4)


## 15.1 Comparando validação cruzada e teste final

Os valores não precisam ser idênticos.

A validação cruzada forneceu uma **estimativa do desempenho esperado**.

O teste final fornece uma avaliação independente em observações que ficaram fora do processo de escolha.


In [ ]:
pr_auc_teste = average_precision_score(y_test, y_prob_final)

comparacao_final = pd.DataFrame({
    "avaliação": [
        "Validação cruzada durante a seleção",
        "Teste final reservado"
    ],
    "PR-AUC": [
        score_cv_final,
        pr_auc_teste
    ]
})

comparacao_final.round(4)


### Pergunta 8

A PR-AUC do teste foi maior ou menor que a média da validação cruzada?

A diferença parece grande?

Por que seria incorreto voltar agora ao Grid Search, modificar os parâmetros olhando o resultado do teste e continuar chamando esse mesmo conjunto de “teste final”?


# 16. Matriz de confusão do modelo final

Além das métricas agregadas, vamos observar os erros concretos.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final,
    display_labels=["Legítima", "Fraude"]
)

plt.title("Modelo final — conjunto de teste")
plt.show()


Observe principalmente:

- **falsos positivos**: operações legítimas sinalizadas como fraude;
- **falsos negativos**: fraudes que o modelo deixou passar.

Em aplicações reais, esses dois erros podem possuir custos muito diferentes.


# 17. Curva Precision-Recall do modelo final


In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_test,
    y_prob_final,
    name="Modelo final"
)

plt.title("Precision-Recall — conjunto de teste")
plt.show()


# 18. O processo completo

A metodologia utilizada nesta aula pode ser resumida assim:

### 1. Reservar o teste final

O teste não participa da escolha do modelo.

### 2. Trabalhar no conjunto de desenvolvimento

Modelos e hiperparâmetros são comparados usando validação cruzada.

### 3. Escolher uma métrica adequada

Neste problema utilizamos PR-AUC como critério principal devido ao forte desbalanceamento.

### 4. Ajustar hiperparâmetros

Grid Search ou Randomized Search procuram configurações promissoras.

### 5. Escolher o modelo

A decisão ocorre antes de consultar o teste.

### 6. Avaliar uma única vez no teste final

Essa etapa estima o desempenho do modelo escolhido em dados independentes.


# 19. Exercício prático

Agora você realizará uma pequena busca para o **Gradient Boosting**.

Use:

```python
parametros_gb = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.03, 0.05, 0.10],
    "max_depth": [2, 3]
}
```

Crie um `GridSearchCV` usando:

- `GradientBoostingClassifier`;
- `scoring="average_precision"`;
- o mesmo objeto `cv`.

**Não utilize o conjunto de teste.**

Ao final, exiba:

- melhores hiperparâmetros;
- melhor PR-AUC média;
- desvio padrão da configuração vencedora.


In [ ]:
# Complete aqui

parametros_gb = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.03, 0.05, 0.10],
    "max_depth": [2, 3]
}



# 20. Desafio — você é responsável pela avaliação

Considere dois modelos hipotéticos:

| Modelo | PR-AUC média | Desvio padrão |
|---|---:|---:|
| Modelo A | 0,742 | 0,018 |
| Modelo B | 0,749 | 0,061 |

Responda:

**1. O Modelo B possui a maior média. Isso é suficiente para escolhê-lo automaticamente?**

**2. O que o maior desvio padrão do Modelo B indica?**

**3. Que outras informações você gostaria de analisar antes da decisão?**

**4. O conjunto de teste poderia ser usado agora para experimentar várias decisões até encontrar a melhor? Por quê?**


# 21. Relatório final da VGAbank

Preencha a recomendação técnica.

**Métrica principal escolhida:** __________________________________

**Modelo escolhido antes do teste:** ______________________________

**PR-AUC média na validação cruzada:** ____________________________

**PR-AUC no teste final:** _______________________________________

**Recall no teste final:** _______________________________________

**A validação cruzada apresentou estabilidade suficiente? Justifique brevemente.**

__________________________________________________________________

**O ajuste de hiperparâmetros trouxe ganho relevante sobre o modelo base?**

__________________________________________________________________

**Por que o teste final não deve participar do ajuste?**

__________________________________________________________________


# 22. Reflexão final

Responda brevemente:

**1. Por que uma única divisão treino/validação pode ser insuficiente?**

**2. O que significa `k=5` em uma validação cruzada?**

**3. Por que utilizamos `StratifiedKFold` neste problema?**

**4. Qual é a diferença entre parâmetro aprendido pelo modelo e hiperparâmetro?**

**5. Qual é a diferença fundamental entre Grid Search e Randomized Search?**

**6. Por que não devemos escolher hiperparâmetros utilizando o conjunto de teste?**

**7. Por que apresentar média e desvio padrão das métricas?**


# 23. Conclusão

Nesta aula, a pergunta deixou de ser simplesmente:

> **Qual modelo obteve o maior número?**

Passamos a perguntar:

> **Como esse desempenho foi estimado? Quanto ele varia? Como o modelo foi escolhido? O teste permaneceu realmente independente?**

O fluxo realizado foi:

**métricas → variabilidade → validação cruzada → comparação → ajuste de hiperparâmetros → escolha → teste final**

Essa sequência é uma parte fundamental da prática científica em aprendizado de máquina.

Um bom resultado não depende apenas de um algoritmo sofisticado.

Depende também de uma **avaliação metodologicamente adequada**.
